In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# Load dataset
df = pd.read_csv('/content/KaggleV2-May-2016.csv')

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


Clean and prepare the dataset

In [2]:
# Convert date columns
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], utc=True)
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], utc=True)

# Create useful date-based features
df['WaitingDays'] = (
    df['AppointmentDay'].dt.date - df['ScheduledDay'].dt.date
).apply(lambda x: x.days)

df['ScheduledWeekday'] = df['ScheduledDay'].dt.weekday
df['AppointmentWeekday'] = df['AppointmentDay'].dt.weekday

# Remove invalid negative ages
df = df[df['Age'] >= 0].copy()

# Remove identifiers and original date columns
X = df.drop(columns=[
    'PatientId', 'AppointmentID',
    'ScheduledDay', 'AppointmentDay', 'No-show'
])

# Convert target: Yes = 1 (missed appointment), No = 0
y = df['No-show'].map({'No': 0, 'Yes': 1})

# Convert categorical columns into numeric columns
X = pd.get_dummies(X, columns=['Gender', 'Neighbourhood'], dtype=int)

# Split once; all three models will use this exact split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Training samples: 88420
Testing samples: 22106
Number of features: 93

Target distribution:
No-show
0    0.798066
1    0.201934
Name: proportion, dtype: float64


Train the three models

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

results = []

# Model 1: Logistic Regression
start = time.time()

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

lr_train_time = time.time() - start

# Model 2: HistGradientBoosting
start = time.time()

hgb_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

hgb_model.fit(X_train, y_train)

hgb_train_time = time.time() - start

print(f"Logistic Regression training time: {lr_train_time:.2f} seconds")
print(f"HistGradientBoosting training time: {hgb_train_time:.2f} seconds")
print("Both models trained successfully!")

Logistic Regression training time: 32.02 seconds
HistGradientBoosting training time: 6.72 seconds
Both models trained successfully!


Train the neural network

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# Define neural network
class AppointmentNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

nn_model = AppointmentNN(X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

# Train
start = time.time()

for epoch in range(10):
    nn_model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = nn_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

nn_train_time = time.time() - start

print(f"\nNeural Network training time: {nn_train_time:.2f} seconds")

Epoch 1/10 - Loss: 0.4674
Epoch 2/10 - Loss: 0.4459
Epoch 3/10 - Loss: 0.4442
Epoch 4/10 - Loss: 0.4430
Epoch 5/10 - Loss: 0.4426
Epoch 6/10 - Loss: 0.4419
Epoch 7/10 - Loss: 0.4404
Epoch 8/10 - Loss: 0.4401
Epoch 9/10 - Loss: 0.4400
Epoch 10/10 - Loss: 0.4392

Neural Network training time: 15.48 seconds


Evaluate all three models

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Generate predictions
lr_pred = lr_model.predict(X_test)
hgb_pred = hgb_model.predict(X_test)

nn_model.eval()
with torch.no_grad():
    nn_outputs = nn_model(X_test_tensor)
    nn_pred = (torch.sigmoid(nn_outputs) >= 0.5).int().numpy().flatten()

# Evaluation function
def evaluate_model(name, actual, predicted, training_time):
    return {
        "Model": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1 Score": f1_score(actual, predicted, zero_division=0),
        "Training Time (sec)": training_time
    }

results = [
    evaluate_model("Logistic Regression", y_test, lr_pred, lr_train_time),
    evaluate_model("HistGradientBoosting", y_test, hgb_pred, hgb_train_time),
    evaluate_model("Neural Network", y_test, nn_pred, nn_train_time)
]

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (sec)
0,Logistic Regression,0.7961,0.3854,0.0166,0.0318,32.0186
1,HistGradientBoosting,0.7992,0.5793,0.0213,0.0411,6.7220
2,Neural Network,0.7982,0.5172,0.0134,0.0262,15.4772


Understanding the results

HistGradientBoosting is currently the best of the three, but its recall is still extremely low.

Why is this happening?

The dataset contains:
- 79.81% appointments attended (No-show = 0)
- 20.19% appointments missed (No-show = 1)

At the default prediction threshold of 0.5, the models are predicting almost every appointment as attended.

Improve the comparison using threshold tuning

In [6]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training:", X_fit.shape)
print("Validation:", X_val.shape)

Training: (70736, 93)
Validation: (17684, 93)


Retrain models using the training split

In [7]:
# Retrain Logistic Regression
lr_tuned = LogisticRegression(max_iter=1000)
lr_tuned.fit(X_fit, y_fit)

# Retrain HistGradientBoosting
hgb_tuned = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)
hgb_tuned.fit(X_fit, y_fit)

# Retrain Neural Network
train_dataset = TensorDataset(
    torch.tensor(X_fit.values, dtype=torch.float32),
    torch.tensor(y_fit.values, dtype=torch.float32).reshape(-1, 1)
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

nn_tuned = AppointmentNN(X_fit.shape[1])
optimizer = torch.optim.Adam(nn_tuned.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(10):
    nn_tuned.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(nn_tuned(inputs), targets)
        loss.backward()
        optimizer.step()

print("All three models retrained successfully!")

All three models retrained successfully!


Find the best threshold

In [8]:
from sklearn.metrics import precision_recall_curve

# Validation predictions (probabilities)
lr_val_prob = lr_tuned.predict_proba(X_val)[:, 1]
hgb_val_prob = hgb_tuned.predict_proba(X_val)[:, 1]

nn_tuned.eval()
with torch.no_grad():
    X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
    nn_val_prob = torch.sigmoid(nn_tuned(X_val_tensor)).numpy().flatten()

# Find best F1 threshold
def find_best_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )

    f1_scores = (
        2 * precision[:-1] * recall[:-1] /
        (precision[:-1] + recall[:-1] + 1e-10)
    )

    best_index = np.argmax(f1_scores)

    return thresholds[best_index], f1_scores[best_index]

thresholds = {}

for name, probabilities in [
    ("Logistic Regression", lr_val_prob),
    ("HistGradientBoosting", hgb_val_prob),
    ("Neural Network", nn_val_prob)
]:
    threshold, best_f1 = find_best_threshold(y_val, probabilities)
    thresholds[name] = threshold

    print(f"{name}:")
    print(f"  Best threshold: {threshold:.4f}")
    print(f"  Validation F1-score: {best_f1:.4f}\n")

Logistic Regression:
  Best threshold: 0.1868
  Validation F1-score: 0.4061

HistGradientBoosting:
  Best threshold: 0.2221
  Validation F1-score: 0.4488

Neural Network:
  Best threshold: 0.2413
  Validation F1-score: 0.4456



Final evaluation on the test set

In [9]:
# Test probabilities
lr_test_prob = lr_tuned.predict_proba(X_test)[:, 1]
hgb_test_prob = hgb_tuned.predict_proba(X_test)[:, 1]

nn_tuned.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
    nn_test_prob = torch.sigmoid(nn_tuned(X_test_tensor)).numpy().flatten()

# Apply validation-selected thresholds
lr_final_pred = (lr_test_prob >= thresholds["Logistic Regression"]).astype(int)
hgb_final_pred = (hgb_test_prob >= thresholds["HistGradientBoosting"]).astype(int)
nn_final_pred = (nn_test_prob >= thresholds["Neural Network"]).astype(int)

# Evaluate
final_results = [
    evaluate_model("Logistic Regression", y_test, lr_final_pred, lr_train_time),
    evaluate_model("HistGradientBoosting", y_test, hgb_final_pred, hgb_train_time),
    evaluate_model("Neural Network", y_test, nn_final_pred, nn_train_time)
]

final_results_df = pd.DataFrame(final_results)
display(final_results_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (sec)
0,Logistic Regression,0.6100,0.2947,0.6682,0.4090,32.0186
1,HistGradientBoosting,0.6238,0.3194,0.7632,0.4504,6.7220
2,Neural Network,0.6519,0.3263,0.6794,0.4408,15.4772


In [10]:
from sklearn.metrics import ConfusionMatrixDisplay

predictions = [
    ("Logistic Regression", lr_final_pred),
    ("HistGradientBoosting", hgb_final_pred),
    ("Neural Network", nn_final_pred)
]

for name, pred in predictions:
    print("\n", name)
    print(confusion_matrix(y_test, pred))


 Logistic Regression
[[10502  7140]
 [ 1481  2983]]

 HistGradientBoosting
[[10383  7259]
 [ 1057  3407]]

 Neural Network
[[11379  6263]
 [ 1431  3033]]


measure model sizes and training times on the final training split

In [11]:
import pickle
import time

# Measure final model sizes
lr_size = len(pickle.dumps(lr_tuned)) / 1024
hgb_size = len(pickle.dumps(hgb_tuned)) / 1024
nn_params = sum(p.numel() for p in nn_tuned.parameters())

print(f"Logistic Regression size: {lr_size:.2f} KB")
print(f"HistGradientBoosting size: {hgb_size:.2f} KB")
print(f"Neural Network parameters: {nn_params:,}")
print(f"Neural Network weights size: {nn_params * 4 / 1024:.2f} KB")

Logistic Regression size: 3.96 KB
HistGradientBoosting size: 300.41 KB
Neural Network parameters: 8,129
Neural Network weights size: 31.75 KB


Final training times and model sizes

In [12]:
import time
import pickle
import io

# 1. Logistic Regression
lr_final = LogisticRegression(max_iter=1000)

start = time.time()
lr_final.fit(X_fit, y_fit)
lr_time = time.time() - start

lr_size = len(pickle.dumps(lr_final)) / 1024

# 2. HistGradientBoosting
hgb_final = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

start = time.time()
hgb_final.fit(X_fit, y_fit)
hgb_time = time.time() - start

hgb_size = len(pickle.dumps(hgb_final)) / 1024

# 3. Neural Network
nn_final = AppointmentNN(X_fit.shape[1])
optimizer = torch.optim.Adam(nn_final.parameters(), lr=0.001)

start = time.time()

for epoch in range(10):
    nn_final.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(nn_final(inputs), targets)
        loss.backward()
        optimizer.step()

nn_time = time.time() - start

buffer = io.BytesIO()
torch.save(nn_final.state_dict(), buffer)
nn_size = len(buffer.getvalue()) / 1024

# Display results
print("FINAL MODEL DETAILS")
print("-" * 40)
print(f"Logistic Regression: {lr_time:.2f}s | {lr_size:.2f} KB")
print(f"HistGradientBoosting: {hgb_time:.2f}s | {hgb_size:.2f} KB")
print(f"Neural Network:       {nn_time:.2f}s | {nn_size:.2f} KB")

FINAL MODEL DETAILS
----------------------------------------
Logistic Regression: 33.48s | 3.96 KB
HistGradientBoosting: 10.28s | 300.41 KB
Neural Network:       12.51s | 34.46 KB
